# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-36: MR-NIRP Indoor Phase-3 Preprocessing

### What this notebook does
Writes the MR-NIRP Indoor dataset into one Phase-3 store in the unified schema (`p3_v1`),
directly in the rechunked uncompressed layout, as an **evaluation-only** corpus for heart rate
and heart-rate variability. Every frame is read once and used three times: the face crop the app
would take, the mean colour of three skin regions, and the crop's own colour trace at the
sensor's full depth. The reference comes from NB_P3_35 - the oximeter's rebuilt clock, the beats
from the steepest upstroke, and each recording's own offset to the camera - and is stored both
on the frame grid and natively.

No estimator is scored here. The POS readings in the store are the app's own, kept as quality
numbers, and the store reproduces NB_P3_35's figures before anything downstream runs.

### Dataset
Nowara, Marks, Mansour, Veeraraghavan, *SparsePPG: Towards Driver Monitoring Using
Camera-Based Vital Signs Estimation in Near-Infrared* (CVPR Workshops 2018, pp. 1272-1281),
distributed by Rice University as the MR-NIRP Indoor dataset, whose README asks that this paper
be cited. Its colour camera is the Blackfly BFLY-U3-23S6C unit (serial 17473548 in the setup
logs) that also recorded the Rice iPPG set, through the same recorder.

### Decisions carried from NB_P3_35 (all measured there)
- **All 15 recordings** carry heart rate; **7 carry HRV**, all of them still recordings. The
  motion recordings are about a minute long and keep 55-62 s of intervals, so they fail the 90 s
  rule on length alone, and one still recording fails on kept share. The list is carried here
  and checked against a recomputation.
- **Frames are dated by the camera's clock** at each recording's own measured rate (29.955 fps,
  29.978 in the two 848x848 recordings), mapped onto the PC's clock through the lower envelope of
  the arrival stamps. A uniform grid at that rate is within 0.045 ms in 14 of 15 recordings.
- **Camera 1 is the colour camera**, identified from the logs; the near-infrared camera's frames
  are not in this copy.
- **The mosaic is RGGB** in all 15 recordings, read from the data - including the one subject
  whose camera the dataset's README gives as `bggr` - which is OpenCV's `COLOR_BayerBG2RGB`.
- **A frame's size is read from its own PGM header**, since two recordings are 848x848 and the
  rest 640x640, and **the frame is converted to the machine's byte order before any demosaic**:
  numpy honours the big-endian order a 16-bit PGM is written in, OpenCV does not.
- **The app's framing gate refuses 13 of 15 recordings** and warns on the other two, because a
  face filling 57-79% of the frame's height cannot fit a box padded by 0.6.

### One crop, not two
The Rice store kept two crops because the choice between them was open. It is no longer: measured
over all 70 of those recordings, the clamped crop - what the app's own box takes - reads more
recordings and reads them at least as closely, and the two are near even paired. This store keeps
that one crop, as `frames`.

### The 8-bit contract, and the trace that measures it
The crops are 8-bit, made the way the app would make them: the high byte of the 12-bit sample
first, then the demosaic, then the crop and resize. That order is not the better one - NB_P3_35
section 8 found the same region's beats timed better at the sensor's full depth in 13 of 15
recordings, by a median 6.6 ms - but it is the app's, and a browser capture never delivers more
than 8 bits. A store that read a pulse the app cannot have would mis-state what the app can do.

So the headroom is kept as what it is: `trace12_rgb`, the same central region of the same crops
averaged at full depth, [T, 3] per recording and a few MB in all. The question stays answerable
from the store rather than from another pass over 48 GB of raw frames, and nothing downstream
mistakes it for a frame.

### Colour
The crops hold raw demosaiced values, with the skin's own balance recorded per recording
(`skin_r_g`, `skin_b_g`). A fixed per-channel gain is not a decision this store has to take:
PhysNet z-scores each channel over the clip and POS divides each channel by its own mean, so
white balance cancels in both. These frames are dark - a median 8-bit level of 31-112 across
recordings - and that is the exposure the dataset was recorded at, so nothing is rescaled: a
per-frame gain would introduce a brightness modulation of its own, in the band a pulse lives in.

### Two guards against this store being trained on
The role decided in NB_P3_35 is an evaluation one. This corpus does have a pulse reference, so
nothing about it is inherently untrainable - which is exactly why the decision is made explicit:
1. **It is written to `phase 3 benchmarks\`, not `phase 3 datasets\`**, outside every training
   glob.
2. **No `cardiac_sqi` attribute.** The training loader skips a group whose `cardiac_sqi` falls
   below 0.05 and reads a missing one as 0.0. The reference's own quality lives in
   `ref_kept_share`, `ref_rmssd_ms` and `has_hrv` instead.

Section 4 applies the loader's own filter to the store and requires that it admit nothing. Adding
this corpus to training later is a decision to take in the open, not by moving a file.

## Imports & Setup
The app's own modules are imported from the repository rather than restated, so every face box,
crop, gate and reading below is the one the app would produce from these frames. The first part
of this cell is NB_P3_35's setup unchanged; the rest is what the store adds - where it goes, the
facts carried from the exploration, and the schema's own constants.

In [1]:
import inspect
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import linprog
from scipy.signal import butter, find_peaks, sosfiltfilt, welch

# Walk up from the notebook until the repo root is found, identified by the MediaPipe asset
# (the same test NB_P3_33 uses; cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The app's code is imported, not restated: every box, crop, gate and reading below is the one
# the app would produce from these frames, and it moves if the app moves. The alignment comes
# from the HRV harness, so a beat offset here means what it means there.
from app.live_vitals.capture.motion import motion_verdict
from app.live_vitals.capture.session import (detection_positions, face_verdict, framing_verdict,
                                             stable_box)
from app.live_vitals.config import (CLIP_LEN, CROP_PADDING, MAX_BOX_SIDE_LOST, MIN_CONFIDENCE,
                                    N_DETECT_FRAMES, SPECTRAL_ROI_FRACTION, TARGET_FPS,
                                    TARGET_SIZE, WINDOW_STRIDE)
from app.live_vitals.estimators.hr_spectral import HRSpectral
from app.live_vitals.preprocess.face_box import make_landmarker
from app.live_vitals.preprocess.frames import crop_resize
from app.live_vitals.scripts.check_hrv_from_video import (MATCH_TOLERANCE_S, align_offset,
                                                          nearest_gaps)
from app.live_vitals.signal.spectral import bandpass, pos, rgb_trace

MRNIRP_ROOT = Path(r'F:\GoogleDriveDatasets\MR_NIPR_Indoor')
RECORDING_DIR = re.compile(r'^Subject(\d+)_(still|motion)_940$')
PGM_HEADER = re.compile(rb'^P5\s+(\d+)\s+(\d+)\s+(\d+)\s', re.S)
# Frame size is read from every file's own header: most recordings are 640x640, and at least
# one subject's are not, which a fixed size would have turned into a decode error or worse.
NOMINAL_FPS = 30.0
CYCLE_WRAP_S = 128.0                         # the camera clock's seconds field wraps every 128 s
SITES = {'00': (0, 0), '01': (0, 1), '10': (1, 0), '11': (1, 1)}   # places in a 2x2 mosaic cell

OX_FS = 60.0                 # the recorder's nominal rate; section 4 measures it
LOSS_LOOKAHEAD = 30          # samples that must all arrive a period late to accept a loss
LOSS_TOL_S = 0.003           # slack on that test, for jitter in the floor itself
CONTROL_LOSS_RATE = 0.01     # injected losses per sample for the positive control
HR_MIN_BPM, HR_MAX_BPM = 35.0, 180.0
BEAT_SPACING = 0.55          # the least a beat may follow the last, as a share of the period
IBI_LOCAL_TOL = 0.25         # an interval this far from the median around it is an artefact
GAP_GUARD = 2                # samples either side of a beat that must be real, not interpolated
HRV_MIN_KEPT = 0.90          # a reference carries HRV with at least this share of its intervals
HRV_MIN_SECONDS = 90.0       # and at least this much of them
FRAME_WORKERS = 8            # frames read, demosaiced and cropped in parallel threads
MAX_RECORDS = None           # set to an int for a trial run

LANDMARKER = make_landmarker()
print('MR-NIRP Indoor root:', MRNIRP_ROOT, '| exists:', MRNIRP_ROOT.exists())
print(f'app contract: crop {TARGET_SIZE}x{TARGET_SIZE} from a box padded by {CROP_PADDING:g}, '
      f'colour trace from its central {SPECTRAL_ROI_FRACTION:.0%}, windows of {CLIP_LEN} frames '
      f'every {WINDOW_STRIDE}, confidence gate {MIN_CONFIDENCE:g}, framing gate '
      f'{MAX_BOX_SIDE_LOST:.0%} of the box side, training rate {TARGET_FPS:g} fps')

import h5py

DATASET_NAME = 'mrnirp_indoor'
# A benchmark folder, not `phase 3 datasets`: the first of the two guards below.
OUT_DIR = Path(r'D:\QualityPhys\phase 3 benchmarks')
OUT_H5 = OUT_DIR / f'{DATASET_NAME}_phase3_rc_none.h5'   # rechunked layout, written directly
LOG_PATH = REPO_ROOT / 'Data' / f'processing_log_{DATASET_NAME}_phase3.csv'   # recording-level only

TASKS = ('still', 'motion')          # the protocol's two conditions, in that order
RGB_CAMERA = 1                       # the colour camera, identified from the logs in NB_P3_35 s1
PATTERN = 'RGGB'                     # the mosaic, read from the data in NB_P3_35 s2
BAYER_CODE = cv2.COLOR_BayerBG2RGB   # OpenCV's code for RGGB: it names the cell from another pixel
COLOUR_SITES = {'R': [(0, 0)], 'G': [(0, 1), (1, 0)], 'B': [(1, 1)]}   # RGGB, (row, column)
SENSOR_CEILING = 65520   # 12 bits left-shifted by four: the largest value the sensor reaches

# Skin regions as fractions of the landmark box (x0, y0, x1, y1), left and right as in the image,
# as NB_P3_33 part 3 placed them: the forehead between the top of the landmarks and the brows,
# each cheek between the eye and the nostril. The region fusion they serve is not settled; the
# traces cost a few MB and save a second pass over 48 GB of frames if it ever is.
REGIONS = {'forehead': (0.30, 0.05, 0.70, 0.20),
           'cheek_left': (0.14, 0.50, 0.36, 0.68),
           'cheek_right': (0.64, 0.50, 0.86, 0.68)}
MIN_REGION_CELLS = 1000              # a region needs this many 2x2 cells inside the frame

MATCH_TOL_S = MATCH_TOLERANCE_S      # a camera beat matches a reference beat this close
# Per-subject anchors, each pooled over that subject's recordings with the HRV harness's wide
# +/-400 ms search. They are carried as a cross-check, not as the centre of the search: a subject
# here has two recordings whose heart rates differ by about a beat per minute, so pooling cannot
# discriminate a slip the way Rice's five activities can, and centring the search on the anchor
# moved 07/motion 137 ms to a worse fit (median beat distance 125.8 to 134.5 ms). Each recording
# is therefore aligned on its own, within +/-250 ms, and `offset_from_anchor_ms` records the
# disagreement. A constant offset does not affect HRV at all - only jitter does - so this choice
# is about what the stored label means, not about the measure this store exists for.
ANCHOR_MS = {'01': 27.5, '02': -10.0, '03': 102.5, '04': -12.5, '05': 45.0, '06': 152.5,
             '07': -87.5, '08': -32.5}
ANCHOR_DISAGREE_MS = 100.0           # beyond this, the recording is flagged rather than trusted
# NB_P3_35 section 7's own offsets, which this store has to reproduce (checked in section 4).
NB_P3_35_OFFSET_MS = {('01', 'motion'): -25.0, ('01', 'still'): 10.0,
                      ('02', 'motion'): 12.5, ('02', 'still'): -17.5,
                      ('03', 'motion'): 62.5, ('03', 'still'): 130.0,
                      ('04', 'motion'): 60.0, ('04', 'still'): -12.5,
                      ('05', 'still'): 45.0,
                      ('06', 'motion'): 155.0, ('06', 'still'): 192.5,
                      ('07', 'motion'): 235.0, ('07', 'still'): -67.5,
                      ('08', 'motion'): -7.5, ('08', 'still'): -75.0}
# The references NB_P3_35 section 5 found able to carry HRV, and checked again in section 4.
HRV_SET = {('01', 'still'), ('02', 'still'), ('03', 'still'), ('04', 'still'),
           ('05', 'still'), ('07', 'still'), ('08', 'still')}

FRAME_DARK, FRAME_BRIGHT = 5.0, 250.0   # the exposure gate, as every Phase-3 store
SCHEMA_VERSION = 'p3_v1'
FRAMES_PER_CHUNK = 32                # frames per HDF5 chunk
OVERWRITE = False

print('output store:', OUT_H5, '(exists)' if OUT_H5.exists() else '(new)')
print(f'carried from NB_P3_35: camera {RGB_CAMERA} is the colour one, the mosaic is {PATTERN}, '
      f'{len(HRV_SET)} of 15 references carry HRV, and every recording keeps its own offset')

MR-NIRP Indoor root: F:\GoogleDriveDatasets\MR_NIPR_Indoor | exists: True
app contract: crop 72x72 from a box padded by 0.6, colour trace from its central 35%, windows of 160 frames every 80, confidence gate 0.65, framing gate 2% of the box side, training rate 30 fps
output store: D:\QualityPhys\phase 3 benchmarks\mrnirp_indoor_phase3_rc_none.h5 (exists)
carried from NB_P3_35: camera 1 is the colour one, the mosaic is RGGB, 7 of 15 references carry HRV, and every recording keeps its own offset


## 1. The reference, as NB_P3_35 built it
The same code as NB_P3_35, restated so this notebook stands alone: the inventory, the frame clock
from the camera's own counter mapped onto the PC's through the lower envelope, the oximeter's
clock rebuilt from its read stamps with the lost samples placed, the waveform on the device's
60 Hz grid, and the beats and interval rules that decide what HRV is computed from.

What the exploration did and this does not: the positive control on the rebuilt clock, the
green-pair test for the mosaic, and the audit tables. Those settled questions, and their answers
are carried in the cell above. What is checked here is that the reference comes out the same -
the heart rates, SDNN, RMSSD and, above all, **which recordings can carry HRV**, against the set
NB_P3_35 named. Nothing in this cell reads a frame; the pass over all 15 recordings takes about a
minute, most of it placing oximeter samples in their slots.

In [2]:
SETTINGS = {'serial': r'Serial number (-?\d+)', 'format': r'img format is: (\w+)',
            'fps_set': r'Frame rate is: ([\d.]+) fps', 'gain_db': r'Gain is: ([\d.]+)',
            'shutter_ms': r'Shutter is: ([\d.]+)'}


def frame_shape(path):
    """(width, height, maxval) from a PGM's own header."""
    with open(path, 'rb') as fh:
        match = PGM_HEADER.match(fh.read(64))
    if match is None:
        raise ValueError(f'{path.name} is not a binary PGM')
    width, height, maxval = (int(g) for g in match.groups())
    return width, height, maxval


def count_lines(path):
    """Non-empty lines in a text log, or -1 when the file is missing."""
    if not path.exists():
        return -1
    with open(path, encoding='utf-8', errors='replace') as fh:
        return sum(1 for line in fh if line.strip())


def camera_settings(path):
    """What the camera reported back in its Setup log."""
    text = path.read_text(encoding='utf-8', errors='replace')
    found = {key: re.search(pattern, text) for key, pattern in SETTINGS.items()}
    return {key: (match.group(1) if match else None) for key, match in found.items()}


def scan_recordings(root=MRNIRP_ROOT):
    """One row per recording: its frame count, both cameras' log lengths and their settings.

    Each recording folder holds a second folder of the same name, and the frames and logs sit
    inside that one. Two cameras were recorded, numbered 0 and 1, and only one of them is the
    colour camera whose frames survive - which camera that is, is read from the data below
    rather than assumed.
    """
    rows = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir() and RECORDING_DIR.match(p.name)):
        subject, task = RECORDING_DIR.match(folder.name).groups()
        inner = folder / folder.name
        frames_dir = inner / 'RGB'
        sizes = [e.stat().st_size for e in os.scandir(frames_dir)
                 if e.name.lower().endswith('.pgm')] if frames_dir.is_dir() else []
        width, height, depth = frame_shape(frames_dir / 'Frame00000.pgm') if sizes else (0, 0, 0)
        row = {'subject': f'{int(subject):02d}', 'task': task, 'folder': inner,
               'frames': len(sizes), 'width': width, 'height': height, 'maxval': depth,
               'sizes_equal': len(set(sizes)) <= 1}
        for index in (0, 1):
            row[f'time_lines{index}'] = count_lines(inner / f'CameraTimeLog{index}.txt')
            row[f'cycle_lines{index}'] = count_lines(inner / f'CameraCyclesLog{index}.txt')
            row[f'frame_lines{index}'] = count_lines(inner / f'CameraFrame{index}.txt')
            settings = camera_settings(inner / f'Setup{index}.log')
            row.update({f'{key}{index}': value for key, value in settings.items()})
        rows.append(row)
    return pd.DataFrame(rows)


def lower_line(x, y):
    """(a, b) of the line a + b*x lying under every point with the least total gap.

    Times written by a PC are late by a variable amount and never early, so a clock relation is
    the lower envelope of the points. With x centred the total gap depends only on the line's
    height at the centre, so the programme maximises that height.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x0, y0 = x.mean(), y.min()
    xc = x - x0
    sol = linprog(c=[-1.0, 0.0], A_ub=np.column_stack([np.ones_like(xc), xc]), b_ub=y - y0,
                  bounds=[(None, None), (None, None)], method='highs')
    if not sol.success:
        raise RuntimeError(f'lower envelope fit failed: {sol.message}')
    a, b = sol.x
    return float(y0 + a - b * x0), float(b)


def read_camera_logs(folder, index=None):
    """Returns (PC arrival times, camera clock from the first frame, frame counter), in s."""
    index = RGB_CAMERA if index is None else index
    host = np.loadtxt(folder / f'CameraTimeLog{index}.txt', ndmin=1)
    cycles = np.loadtxt(folder / f'CameraCyclesLog{index}.txt', ndmin=1)
    with open(folder / f'CameraFrame{index}.txt', encoding='utf-8', errors='replace') as fh:
        counter = np.array([int(re.search(r'\d+', line).group()) for line in fh if line.strip()])
    step = np.diff(cycles)
    step[step < 0] += CYCLE_WRAP_S           # the seconds field wraps
    return host, np.concatenate([[0.0], np.cumsum(step)]), counter


def frame_clock(folder):
    """Frame times on the PC clock, and the frame rate on the camera's own clock."""
    host, clock, _ = read_camera_logs(folder)
    a, b = lower_line(clock, host - clock)
    return a + (1.0 + b) * clock, (len(clock) - 1) / float(clock[-1])


def clock_for_frames(folder, frames):
    """The frame times of the saved frames, where the logs may run a few lines longer."""
    times, fps = frame_clock(folder)
    return times[:frames], fps


def timing_audit(row):
    """Drops, rate, grid deviation and the PC's lateness for one recording."""
    host, clock, counter = read_camera_logs(row.folder)
    a, b = lower_line(clock, host - clock)
    lag_ms = (host - (a + (1.0 + b) * clock)) * 1000.0
    step_ms = np.diff(clock) * 1000.0
    counter_step = np.diff(counter)
    index = np.arange(clock.size)
    grid_ms = (clock - np.polyval(np.polyfit(index, clock, 1), index)) * 1000.0
    return {'subject': row.subject, 'task': row.task, 'frames': len(host),
            'dropped': int((counter_step[counter_step > 0] - 1).sum()),
            'backwards': int((counter_step <= 0).sum()),
            'fps': 1000.0 / float(np.median(step_ms)),
            'step_min_ms': float(step_ms.min()), 'step_max_ms': float(step_ms.max()),
            'grid_max_ms': float(np.abs(grid_ms).max()), 'span_s': float(clock[-1]),
            'drift_ppm': b * 1e6, 'lag_med_ms': float(np.median(lag_ms)),
            'lag_max_ms': float(lag_ms.max()), 'lag_over_10ms': float((lag_ms > 10).mean())}


def read_oximeter(folder):
    """The waveform and the PC's read time of each sample, from pulseOx.mat.

    The record is stored as a ragged object array here and as a plain integer array in the car
    dataset from the same group, so it is flattened rather than converted element by element.
    Its values carry no offset in this dataset, where the car's and Rice's carry the top bit
    set, so an offset is removed only where the data shows one.
    """
    mat = sio.loadmat(folder / 'PulseOX' / 'pulseOx.mat', squeeze_me=True)
    record = np.atleast_1d(mat['pulseOxRecord'])
    wave = np.concatenate([np.atleast_1d(np.asarray(v)).ravel() for v in record]).astype(float)
    stamps = np.atleast_1d(mat['pulseOxTime']).astype(np.float64)
    claimed = int(np.atleast_1d(mat['numPulseSample'])[0])
    offset = 128.0 if wave.min() >= 128 else 0.0
    return wave - offset, stamps, claimed, offset


def sample_slots(stamps, fs=OX_FS, lookahead=LOSS_LOOKAHEAD, tol=LOSS_TOL_S):
    """Places every read sample in its slot on the device's own grid (NB_P3_33 section 3).

    slots[k] is k plus the samples lost before sample k. A stamp minus its slot time sits on a
    floor of prompt reads, and the floor steps up one period at each lost sample. A loss is
    accepted only when all of the next `lookahead` samples sit at least a period (less `tol`)
    above the floor of the `lookahead` before them, so one late read never passes for a loss.
    """
    period = 1.0 / fs
    n = stamps.size
    slots = np.empty(n, dtype=np.int64)
    floor = np.empty(n)
    below = np.zeros(n, dtype=bool)
    lost = 0
    for k in range(n):
        hi = min(n, k + lookahead)
        ahead = stamps[k:hi] - (np.arange(k, hi) + lost) * period
        before = ahead.min() if k == 0 else floor[max(0, k - lookahead):k].min()
        while hi - k >= 5 and ahead.min() - before >= period - tol:
            lost += 1
            ahead = ahead - period
        below[k] = ahead.min() - before <= -(period - tol)
        slots[k] = k + lost
        floor[k] = stamps[k] - slots[k] * period
    steps_down = int(below[0]) + int((below[1:] & ~below[:-1]).sum())
    return slots, steps_down


def rebuild_clock(stamps, fs=OX_FS):
    """Slots, sample times on the PC clock, read delays and the device's measured rate."""
    slots, steps_down = sample_slots(stamps, fs)
    grid_s = slots / fs
    a, b = lower_line(grid_s, stamps - grid_s)
    times = a + (1.0 + b) * grid_s
    return {'slots': slots, 'times': times, 'delay': stamps - times,
            'rate_hz': fs / (1.0 + b), 'steps_down': steps_down}


BEAT_BAND = butter(2, [0.5, 8.0], btype='band', fs=OX_FS, output='sos')


HR_BAND = butter(2, [HR_MIN_BPM / 60.0, HR_MAX_BPM / 60.0], btype='band', fs=OX_FS, output='sos')


def on_grid(wave, slots):
    """The waveform on the device's own grid, with lost samples interpolated and marked."""
    grid = np.full(int(slots[-1]) + 1, np.nan)
    grid[slots[:wave.size]] = wave[:slots.size]
    missing = np.isnan(grid)
    if missing.any():
        idx = np.arange(grid.size)
        grid[missing] = np.interp(idx[missing], idx[~missing], grid[~missing])
    return grid, missing


def dominant_period(signal, fs):
    """The dominant beat period in samples, from a 16 s Welch spectrum."""
    f, p = welch(signal - signal.mean(), fs=fs, nperseg=min(len(signal), int(16 * fs)))
    band = (f >= HR_MIN_BPM / 60.0) & (f <= HR_MAX_BPM / 60.0)
    return fs / f[band][np.argmax(p[band])]


def refine(y, i):
    """Sub-sample position of a maximum from a parabola through three points."""
    if i <= 0 or i >= len(y) - 1:
        return float(i)
    left, centre, right = float(y[i - 1]), float(y[i]), float(y[i + 1])
    denominator = left - 2.0 * centre + right
    if abs(denominator) < 1e-12:
        return float(i)
    off = 0.5 * (left - right) / denominator
    return float(i) + (off if abs(off) <= 0.5 else 0.0)


def steepest_rises(signal, fs, period):
    """Sub-sample positions of the steepest upstrokes: NB_P3_33's fiducial and rules."""
    slope = np.gradient(signal)
    idx, _ = find_peaks(slope, distance=max(1, int(BEAT_SPACING * period)))
    if idx.size < 3:
        return np.array([]), slope
    local = pd.Series(slope[idx]).rolling(9, center=True, min_periods=3).median().to_numpy()
    idx = idx[slope[idx] >= 0.4 * local]
    return np.array([refine(slope, int(i)) for i in idx]), slope


def find_beats(grid, missing, fs=OX_FS):
    """One row per beat: its steepest-upstroke time, and whether it borders a lost sample."""
    x = sosfiltfilt(BEAT_BAND, grid - grid.mean())
    period = dominant_period(sosfiltfilt(HR_BAND, grid - grid.mean()), fs)
    positions, _ = steepest_rises(x, fs, period)
    rows = []
    for p in positions:
        i = int(round(p))
        around = slice(max(0, i - GAP_GUARD), i + GAP_GUARD + 1)
        rows.append({'slope_s': p / fs, 'near_gap': bool(missing[around].any()),
                     'clipped': bool(((grid[around] <= grid.min()) |
                                      (grid[around] >= grid.max())).any())})
    return pd.DataFrame(rows, columns=['slope_s', 'near_gap', 'clipped']), period


def intervals(beats):
    """Beat-to-beat intervals with NB_P3_33's artefact rules applied."""
    t = beats.slope_s.to_numpy()
    ibi = np.diff(t)
    flagged = (beats.near_gap | beats.clipped).to_numpy()
    in_range = (ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)
    clean = ~flagged[:-1] & ~flagged[1:]
    candidate = in_range & clean
    local = (pd.Series(np.where(candidate, ibi, np.nan))
             .rolling(7, center=True, min_periods=3).median().to_numpy())
    consistent = np.abs(ibi - local) <= IBI_LOCAL_TOL * local
    return pd.DataFrame({'t': t[1:], 'ibi': ibi, 'kept': candidate & consistent})


def hrv_metrics(table):
    """Mean heart rate, SDNN and RMSSD from the kept intervals."""
    ibi, kept = table.ibi.to_numpy(), table.kept.to_numpy()
    pair = kept[1:] & kept[:-1]
    diffs, values = np.diff(ibi)[pair], ibi[kept]
    if values.size < 8 or diffs.size < 8:
        return dict.fromkeys(('hr_bpm', 'sdnn_ms', 'rmssd_ms'), np.nan)
    return {'hr_bpm': 60.0 / values.mean(), 'sdnn_ms': values.std(ddof=1) * 1000.0,
            'rmssd_ms': float(np.sqrt(np.mean(diffs ** 2)) * 1000.0)}


recordings = scan_recordings()
recordings['log_lines'] = recordings[f'time_lines{RGB_CAMERA}']

REF = {}
rows = []
for r in recordings.itertuples():
    key = (r.subject, r.task)
    clocks = timing_audit(r)
    frame_t, _ = clock_for_frames(r.folder, int(r.frames))
    # The rate of the frames the store keeps, so that fps times duration returns the frame count:
    # a log can run a line longer than its frames, and one recording stalls for a quarter second,
    # which the rate over the whole span absorbs and the median frame interval does not.
    fps = (frame_t.size - 1) / float(frame_t[-1] - frame_t[0])
    wave, stamps, claimed, offset_removed = read_oximeter(r.folder)
    clock = rebuild_clock(stamps)
    grid, missing = on_grid(wave, clock['slots'])
    grid_t = np.interp(np.arange(grid.size), clock['slots'], clock['times'])
    beats, period = find_beats(grid, missing)
    table = intervals(beats)
    metrics = hrv_metrics(table)
    beat_t = np.interp(beats.slope_s.to_numpy() * OX_FS, clock['slots'], clock['times'])
    gaps = np.diff(clock['slots']) - 1
    kept_s = float(table.loc[table.kept, 'ibi'].sum())
    carries = bool(table.kept.mean() >= HRV_MIN_KEPT and kept_s >= HRV_MIN_SECONDS)
    REF[key] = {'frame_t': frame_t, 'fps': fps, 'grid': grid, 'grid_t': grid_t,
                'missing': missing, 'beat_t': beat_t, 'beat_kept': table.kept.to_numpy(),
                'metrics': metrics, 'row': r, 'timing': clocks,
                'ox': {'samples': int(stamps.size), 'n_values': int(wave.size),
                       'claimed': claimed, 'offset_removed': offset_removed,
                       'rate_hz': clock['rate_hz'], 'lost': int(gaps.sum()),
                       'longest_gap': int(gaps.max()) if gaps.size else 0,
                       'delay_med_ms': float(np.median(clock['delay']) * 1000.0),
                       'starts_after_s': float(clock['times'][0] - frame_t[0]),
                       'short_by_s': float(frame_t[-1] - clock['times'][-1])},
                'kept_share': float(table.kept.mean()), 'kept_s': kept_s, 'carries_hrv': carries}
    rows.append({'subject': r.subject, 'task': r.task, 'frames': int(r.frames),
                 'fps': fps, 'fps_step': clocks['fps'], 'grid_max_ms': clocks['grid_max_ms'],
                 'dropped': clocks['dropped'], 'samples': int(stamps.size),
                 'lost': int(gaps.sum()),
                 'beats': len(beats), 'kept_share': float(table.kept.mean()), 'kept_s': kept_s,
                 'short_by_s': REF[key]['ox']['short_by_s'], **metrics,
                 'carries_hrv': carries})
reference = pd.DataFrame(rows)

mismatch = {k for k in REF if REF[k]['carries_hrv']} ^ HRV_SET
print(f'references built: {len(REF)} | carrying HRV: {int(reference.carries_hrv.sum())} '
      f'(NB_P3_35: {len(HRV_SET)}) | disagreement with the carried set: {mismatch or "none"}')
print(f'heart rate {reference.hr_bpm.min():.1f}-{reference.hr_bpm.max():.1f} bpm | SDNN '
      f'{reference.sdnn_ms.min():.1f}-{reference.sdnn_ms.max():.1f} ms | RMSSD '
      f'{reference.rmssd_ms.min():.1f}-{reference.rmssd_ms.max():.1f} ms | intervals kept, '
      f'median {reference.kept_share.median():.1%}, lowest {reference.kept_share.min():.1%}')
print(f'frame rate {reference.fps.min():.3f}-{reference.fps.max():.3f} fps | frames dropped '
      f'{int(reference.dropped.sum())}'
      " | a uniform grid at the measured rate is within "
      f'{reference.grid_max_ms.max():.1f} ms at worst | the reference stops '
      f'{reference.short_by_s.max():.1f} s before the last frame at worst')
far = reference.loc[(reference.fps - reference.fps_step).abs() > 0.01]
print(f'the rate over a whole span and the median frame interval disagree by more than 0.01 fps '
      f'in {len(far)} recording(s)' + (': ' + ', '.join(
          f'{r.subject}/{r.task} {r.fps:.3f} against {r.fps_step:.3f}'
          for r in far.itertuples()) if len(far) else '')
      + ' - the span is what a frequency reading needs, so it is what the store keeps as fps')
print()
print(reference.round(2).to_string(index=False))

references built: 15 | carrying HRV: 7 (NB_P3_35: 7) | disagreement with the carried set: none
heart rate 53.5-81.3 bpm | SDNN 38.8-81.9 ms | RMSSD 25.7-95.0 ms | intervals kept, median 96.5%, lowest 87.8%
frame rate 29.955-29.997 fps | frames dropped 0 | a uniform grid at the measured rate is within 128.9 ms at worst | the reference stops 14.3 s before the last frame at worst
the rate over a whole span and the median frame interval disagree by more than 0.01 fps in 1 recording(s): 03/motion 29.997 against 29.955 - the span is what a frequency reading needs, so it is what the store keeps as fps

subject   task  frames   fps  fps_step  grid_max_ms  dropped  samples  lost  beats  kept_share  kept_s  short_by_s  hr_bpm  sdnn_ms  rmssd_ms  carries_hrv
     01 motion    1966 29.96     29.96         0.04        0     3910    19     86        0.96   61.68        0.01   79.76    51.25     43.22        False
     01  still    5381 29.96     29.96         0.04        0    10703    52    241     

## 2. Store schema and the single-recording walkthrough

### Schema (MR-NIRP Indoor)
| name | source | shape, type | notes |
|------|--------|-------------|-------|
| `frames` | raw frames, box clamped | [T, 72, 72, 3] uint8 | the app's own crop, from the 8-bit image |
| `bvp` | oximeter waveform | [T] float32 | on the frame grid, with `offset_ms` applied: the label |
| `bvp_covered` | - | [T] bool | false where the oximeter had stopped, or had not started |
| `region_rgb` | raw frames, three regions | [T, 3, 3] float32 | forehead, left cheek, right cheek; mean R, G, B from the mosaic |
| `trace12_rgb` | raw frames, crop centre | [T, 3] float32 | the same region `rgb_trace` reads, at the sensor's full depth |
| `frame_t_ms` | camera clock | [T] float64 | ms after the first frame, on the PC clock |
| `ox_ppg`, `ox_t_ms`, `ox_missing` | oximeter | [N] each | the native 60 Hz record, its rebuilt clock, and which samples were interpolated |
| `ref_beat_ms`, `ref_beat_kept` | oximeter | [B], [B-1] | reference beat times, and which intervals the rules kept |

Key attributes: `role='heldout'`, `activity`, `has_hrv`; `fps` from the camera clock beside
`fps_nominal`, and `source_w`/`source_h` because the recordings are not all one size; the crop
box, `box_side_lost` (the store convention, worse axis) beside `app_frac_side_lost` (what the
app's framing gate reads); the encoding (`bayer_pattern`, `demosaic`, `sensor_bits`,
`container_bits`, `byte_order`, `clipped_share`); `grid_max_ms`, `pc_lag_med_ms` and
`pc_lag_over_10ms`, so the one recording whose frame spacing and arrival stamps misbehave is
flagged in the store rather than remembered; `offset_ms` with its median beat distance, matched
share, slip margin and disagreement with the subject's anchor; the reference's own numbers
(`ref_hr_bpm`, `ref_sdnn_ms`, `ref_rmssd_ms`, `ref_kept_share`, `ref_short_by_s`); the app's
verdicts (`framing_verdict`, `face_checks`, `motion_verdict`, `motion_crop_px`); and the quality
readings `qc_pos_hr`, `qc_pos_hr_error` and `qc_pos_bvp_r`/`qc_pos_bvp_lag_s`. There is **no
`cardiac_sqi`** - see the guards above.

`usable` gates on exposure only, as in every Phase-3 store. A recording the framing gate refuses
is still written, flagged: every recording in this set is one, and refusing to store them would
leave nothing.

### Nothing is trimmed
One recording's reference stops 14.3 s before its last frame, and one more stops half a second
early. Those frames are kept and marked by `bvp_covered`, which is what every other Phase-3 store
does and what the evaluation scripts already read; trimming would cost the frame count its
agreement with the camera's own logs for 7 MB of frames. An evaluation scores the frames the
reference covers, and `ref_short_by_s` says how many it does not.

### The alignment, and why it is not pooled here
Each recording's offset is searched on its own, within 250 ms, by the HRV harness's
`align_offset` - the offset at which the median distance from each reference beat to the nearest
camera beat is smallest - and reported with the margin by which it beats the best rival
alignment more than 250 ms away. The Rice store instead centred each search on a subject anchor
pooled over that subject's five activities, which is what makes a whole-beat slip impossible to
mistake: the activities' heart rates differ, so a slip fits each one at a different shift.

That does not transfer. A subject here has two recordings, still and moving, whose heart rates
differ by about one beat per minute, so pooling cannot discriminate - and measured, it does not:
centring the search on the pooled anchor moved one recording's offset 137 ms to a **worse** fit
(median beat distance 125.8 to 134.5 ms). The anchors are still computed and carried, as
`offset_anchor_ms`, and a recording whose own offset sits more than 100 ms from its subject's
anchor is flagged by `offset_agrees_with_anchor` rather than moved. A constant offset does not
affect HRV at all - only jitter does - so what this choice decides is what the stored label
means, not the measure the store exists for.

### The label's polarity
`qc_pos_bvp_r` is the most negative correlation between the app's POS and the stored label within
0.25 s, with its lag. POS correlates negatively with a stored `bvp` in every Phase-3 store, so
the search is for that sign: the opposite-sign lobe half a beat away reaches the edge of the
window with nearly the same magnitude, and taking the larger magnitude has twice made a sound
label look flipped.

In [3]:
def read_raw(path):
    """One frame as native-order uint16, with its size and header length read from the file.

    A 16-bit PGM is big-endian by the format's definition, and the copy into the machine's own
    order is not a formality: numpy honours a byte order, OpenCV does not. Handed a big-endian
    array, cv2.cvtColor reads the same bytes as little-endian and demosaics the two halves of
    every pixel swapped, which on these frames raised the mean from 21087 to 27953 and left a
    picture that still looks like a face. The conversion costs one copy per frame and removes
    the trap for every caller. The pixel count is checked against the header, so a short or
    padded file raises here rather than reshaping into a skewed image.
    """
    data = path.read_bytes()
    match = PGM_HEADER.match(data[:64])
    if match is None:
        raise ValueError(f'{path.name} is not a binary PGM')
    width, height, _ = (int(g) for g in match.groups())
    end = match.end()
    if len(data) - end != width * height * 2:
        raise ValueError(f'{path.name} holds {len(data) - end} bytes of pixels, not '
                         f'{width * height * 2}')
    return np.frombuffer(data, dtype='>u2', offset=end,
                         count=width * height).reshape(height, width).astype(np.uint16)


def to_8bit(raw):
    """The high byte of each sample, which is the 8-bit image.

    The container is 16 bits wide whatever the sensor's depth, so the 8-bit image is the top
    byte. Shifting by 4 instead leaves 12 bits, and casting that to uint8 wraps modulo 256 into
    noise that still has frame-like statistics - a first probe of this dataset reported a face
    in none of its samples for exactly that reason. Per-frame rescaling is also avoided: it
    would introduce a brightness modulation of its own, in the band a pulse lives in.
    """
    return (raw >> 8).astype(np.uint8)


def frame_path(folder, index):
    return folder / 'RGB' / f'Frame{index:05d}.pgm'


def demosaic(raw):
    """The 8-bit colour image: the high byte first, then the mosaic, as the app's contract asks.

    The order is the app's, not the better one. A browser hands the app 8-bit frames, so a store
    that demosaiced at the sensor's 12 bits and shifted afterwards would read a pulse the app
    cannot have (NB_P3_35 section 8 measures the difference: beats timed better in 13 of 15
    recordings, by a median 6.6 ms). The deeper trace is kept separately, as a trace, for
    exactly that comparison.
    """
    return cv2.cvtColor(to_8bit(raw), BAYER_CODE)


def box_audit(row):
    """The app's held box and its own gates on one recording's detection samples."""
    samples = [demosaic(read_raw(frame_path(row.folder, i)))
               for i in detection_positions(int(row.frames))]
    box, found, faces = stable_box(samples, LANDMARKER)
    if box is None:
        raise RuntimeError('no face in any detection sample')
    framing, report, _ = framing_verdict(box, int(row.width), int(row.height))
    checks, _, summary = face_verdict(faces, box)
    subject = np.array([f[0] for f in faces if f])
    return box, {'found': int(found), 'of': len(samples),
                 'face_w': float(np.median(subject[:, 2] - subject[:, 0])),
                 'face_h': float(np.median(subject[:, 3] - subject[:, 1])),
                 'framing': framing, 'app_frac_side_lost': float(report['frac_side_lost']),
                 'aspect_in_training_range': bool(report['aspect_in_training_range']),
                 'face_checks': checks, 'second_face': int(summary['second_face_samples']),
                 'face_inside_min': float(summary['face_inside_min'])}


def box_side_lost(box):
    """The share of the box's side that falls outside the frame, on the worse axis - the store
    convention, each axis against its own side."""
    lost_x = max(0.0, -box[0]) + max(0.0, box[2] - 1.0)
    lost_y = max(0.0, -box[1]) + max(0.0, box[3] - 1.0)
    return float(max(lost_x / (box[2] - box[0]), lost_y / (box[3] - box[1])))


def region_rectangles(box, face_w, face_h, width, height):
    """Each region's rows and columns on the mosaic, on whole 2x2 cells, or None where fewer than
    MIN_REGION_CELLS cells fall inside the frame. The regions sit on the landmark box: centred
    where the held box is centred, with the median landmark width and height of the samples."""
    left = (box[0] + box[2]) / 2 - face_w / 2
    top = (box[1] + box[3]) / 2 - face_h / 2
    out = {}
    for name, (a, b, c, d) in REGIONS.items():
        c0, c1 = (int(np.clip((left + f * face_w) * width, 0, width)) // 2 * 2 for f in (a, c))
        r0, r1 = (int(np.clip((top + f * face_h) * height, 0, height)) // 2 * 2 for f in (b, d))
        cells = max(0, r1 - r0) * max(0, c1 - c0) // 4
        out[name] = (slice(r0, r1), slice(c0, c1)) if cells >= MIN_REGION_CELLS else None
    return out


def region_colours(raw, rectangles):
    """Mean R, G and B of each region straight from the mosaic, each colour over its own sites. A
    demosaiced frame interpolates each colour from the same sites, so over thousands of cells the
    means agree, without the interpolation's edges. Taken at the sensor's full depth: these are
    traces, not frames, so the app's 8-bit contract does not apply to them."""
    out = np.full((len(rectangles), 3), np.nan)
    for k, place in enumerate(rectangles.values()):
        if place is None:
            continue
        patch = raw[place]
        for c, sites in enumerate(COLOUR_SITES.values()):
            out[k, c] = np.mean([patch[s[0]::2, s[1]::2].mean() for s in sites])
    return out


def frame_pass(row, box, rectangles):
    """One read per frame, three uses: the app's 8-bit crop, the regions' colours, and the crop's
    own colour trace at the sensor's full depth.

    Frames are read, demosaiced and cropped in parallel threads: file reads and OpenCV's calls
    release the interpreter's lock. The two demosaics of a frame are the price of keeping the
    app's contract exactly - 8 bits first, then colour - while still measuring what that contract
    costs; NB_P3_35 section 8 is the reason the deeper trace is here at all.
    """
    n = int(row.frames)
    frames = np.empty((n, TARGET_SIZE, TARGET_SIZE, 3), dtype=np.uint8)
    regions = np.empty((n, len(rectangles), 3), dtype=np.float32)
    trace = np.empty((n, 3), dtype=np.float32)

    def one(i):
        raw = read_raw(frame_path(row.folder, i))
        crop = crop_resize(demosaic(raw), box)
        deep = crop_resize(cv2.cvtColor(raw, BAYER_CODE), box)
        return i, crop, region_colours(raw, rectangles), rgb_trace(deep[None].astype(np.float64))[0]

    with ThreadPoolExecutor(FRAME_WORKERS) as pool:
        for i, crop, colours, deep in pool.map(one, range(n)):
            frames[i], regions[i], trace[i] = crop, colours, deep
    return frames, regions, trace


def camera_beats_on_clock(trace, fps, frame_t):
    """The app's POS pulse from a colour trace, inverted so the systolic rise is a rising edge,
    and the PC times of its steepest rises."""
    pulse = -pos(trace, fps)
    if not np.any(pulse):
        return pulse, np.array([])
    positions, _ = steepest_rises(pulse, fps, dominant_period(pulse, fps))
    return pulse, np.interp(positions, np.arange(frame_t.size), frame_t)


def signed_lag(video, reference, fps, max_lag_s=0.25):
    """(r, lag in s) at the lag within +/-max_lag_s where r is most negative.

    POS and a stored bvp have opposite polarity in every Phase-3 store, so the matching
    alignment is the most negative correlation. Taking the largest magnitude instead is
    ambiguous: the opposite-sign lobe half a beat away reaches the edge of the search with
    nearly the same magnitude, which is how it has twice been mistaken for a flipped label.
    """
    span = int(round(max_lag_s * fps))
    lags = np.arange(-span, span + 1)
    rs = np.array([np.corrcoef(video[max(0, -k):len(video) - max(0, k)],
                               reference[max(0, k):len(reference) - max(0, -k)])[0, 1]
                   for k in lags])
    if np.all(np.isnan(rs)):
        return float('nan'), float('nan')
    i = int(np.nanargmin(rs))
    return float(rs[i]), float(lags[i] / fps)


def reading_of(frames_u8, fps):
    """The app's own heart-rate reading from a crop, as a flat dictionary."""
    result = HRSpectral().estimate(frames_u8, fps)
    detail = result.detail or {}
    return {'hr': float(result.value), 'status': result.status,
            'confidence': float(result.confidence),
            'usable_fraction': float(detail.get('usable_fraction', 0.0)),
            'kept': int(sum(detail.get('window_kept', []))),
            'windows': int(detail.get('n_total', 0))}


def plain(value):
    """A value HDF5 attributes accept: numpy scalars become Python ones, None becomes NaN."""
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value)
    return float('nan') if value is None else str(value)


def clipped_share(row, samples=12):
    """The share of pixels at the sensor's ceiling, over frames spread across the recording.

    Read from the raw values, not the 8-bit ones: the high byte maps the container's top 256
    codes onto 255, which overstates saturation several-fold (11.4% against 0.6% on this set).
    """
    positions = np.linspace(0, int(row.frames) - 1, samples).astype(int)
    return float(np.mean([(read_raw(frame_path(row.folder, int(i))) >= SENSOR_CEILING).mean()
                          for i in positions]))


def build_record(key):
    """Builds one unified-schema record for an MR-NIRP Indoor recording."""
    t_start = time.perf_counter()
    ref = REF[key]
    row = ref['row']
    frame_t, fps = ref['frame_t'], ref['fps']
    box, gates = box_audit(row)
    rectangles = region_rectangles(box, gates['face_w'], gates['face_h'],
                                   int(row.width), int(row.height))
    frames, regions, trace12 = frame_pass(row, box, rectangles)
    if len(frames) != frame_t.size:
        raise RuntimeError(f'{len(frames)} frames against {frame_t.size} clock entries')

    app_read = reading_of(frames, fps)
    motion, motion_note, motion_summary = motion_verdict(frames, fps)
    pulse, cam_beats = camera_beats_on_clock(rgb_trace(frames), fps, frame_t)
    offset, gap_s, matched, margin_ms = align_offset(cam_beats, ref['beat_t'])
    anchor = ANCHOR_MS[key[0]] / 1000.0
    found = bool(cam_beats.size >= 3 and np.isfinite(gap_s))
    used = offset if found else anchor

    # The reference on the frame grid: the finger's waveform at each frame's time plus the offset,
    # which is where that frame's beat is recorded at the finger.
    bvp = np.interp(frame_t + used, ref['grid_t'], ref['grid']).astype(np.float32)
    covered = (frame_t + used >= ref['grid_t'][0]) & (frame_t + used <= ref['grid_t'][-1])
    readout_band = butter(4, [HR_MIN_BPM / 60.0, 3.5], btype='band', fs=fps, output='sos')
    # POS as the app computes it, not the inverted copy the beats are found on, so the sign of r
    # means the same here as in every other Phase-3 store.
    pos_r, pos_lag = signed_lag(sosfiltfilt(readout_band, -pulse),
                                sosfiltfilt(readout_band, bvp - bvp.mean()), fps)
    brightness = float(np.median([frames[i].mean()
                                  for i in np.linspace(0, len(frames) - 1, 9).astype(int)]))
    skin = regions.reshape(-1, 3)
    skin = np.nanmedian(skin[np.isfinite(skin).all(axis=1)], axis=0)

    arrays = {'frames': frames, 'bvp': bvp, 'bvp_covered': covered, 'region_rgb': regions,
              'trace12_rgb': trace12,
              'frame_t_ms': ((frame_t - frame_t[0]) * 1000.0).astype(np.float64),
              'ox_ppg': ref['grid'].astype(np.float32),
              'ox_t_ms': ((ref['grid_t'] - frame_t[0]) * 1000.0).astype(np.float64),
              'ox_missing': ref['missing'],
              'ref_beat_ms': ((ref['beat_t'] - frame_t[0]) * 1000.0).astype(np.float64),
              'ref_beat_kept': ref['beat_kept']}
    attrs = dict(
        schema_version=SCHEMA_VERSION, dataset='MR-NIRP-Indoor', role='heldout',
        subject_id=key[0], subject_uid=f'mrnirp_indoor_{key[0]}', activity=key[1],
        source_dir=row.folder.name,
        fps=float(fps), fps_nominal=NOMINAL_FPS, fps_median_step=float(ref['timing']['fps']),
        n_frames=int(len(frames)),
        frame_size=int(TARGET_SIZE), source_w=int(row.width), source_h=int(row.height),
        duration_s=float(frame_t[-1] - frame_t[0]),
        camera_frames_dropped=int(ref['timing']['dropped']),
        grid_max_ms=float(ref['timing']['grid_max_ms']),
        pc_lag_med_ms=float(ref['timing']['lag_med_ms']),
        pc_lag_over_10ms=float(ref['timing']['lag_over_10ms']),
        crop_rule='the app box clamped to the frame, from the 8-bit image',
        box_rx0=float(box[0]), box_ry0=float(box[1]), box_rx1=float(box[2]), box_ry1=float(box[3]),
        box_side_lost=box_side_lost(box),
        bayer_pattern=PATTERN, demosaic='cv2.COLOR_BayerBG2RGB',
        sensor_bits=12, container_bits=16,
        byte_order='big-endian in the file, converted to native before any demosaic',
        clipped_share=clipped_share(row),
        skin_r=float(skin[0]), skin_g=float(skin[1]), skin_b=float(skin[2]),
        skin_r_g=float(skin[0] / skin[1]), skin_b_g=float(skin[2] / skin[1]),
        regions=','.join(REGIONS), regions_read=int(sum(p is not None
                                                        for p in rectangles.values())),
        has_frames=True, has_bvp=True, has_region_rgb=True, has_trace12=True,
        has_hrv=bool(ref['carries_hrv']),
        has_ecg=False, has_resp=False, has_spo2=False, has_eda=False, has_hr=False,
        bvp_units_note='CMS50D+ finger waveform, 0-112 with no offset removed, on its rebuilt '
                       '60 Hz clock, interpolated to the frame times with offset_ms applied',
        trace12_note='the same central region of the same crops as rgb_trace reads, at the '
                     "sensor's full depth: the headroom of NB_P3_35 section 8, not a frame",
        ox_fs=float(OX_FS), ox_rate_hz=float(ref['ox']['rate_hz']),
        ox_samples=int(ref['ox']['samples']), ox_values=int(ref['ox']['n_values']),
        ox_claimed=int(ref['ox']['claimed']), ox_offset_removed=float(ref['ox']['offset_removed']),
        ox_lost=int(ref['ox']['lost']), ox_longest_gap=int(ref['ox']['longest_gap']),
        ox_delay_med_ms=float(ref['ox']['delay_med_ms']),
        ref_starts_after_s=float(ref['ox']['starts_after_s']),
        ref_short_by_s=float(ref['ox']['short_by_s']),
        ref_covered_frames=int(covered.sum()),
        ref_beats=int(ref['beat_t'].size), ref_kept_share=float(ref['kept_share']),
        ref_kept_s=float(ref['kept_s']),
        ref_hr_bpm=float(ref['metrics']['hr_bpm']), ref_sdnn_ms=float(ref['metrics']['sdnn_ms']),
        ref_rmssd_ms=float(ref['metrics']['rmssd_ms']),
        offset_ms=float(used * 1000.0), offset_found=found,
        offset_gap_ms=float(gap_s * 1000.0), offset_matched=float(matched),
        offset_margin_ms=float(margin_ms), offset_anchor_ms=float(ANCHOR_MS[key[0]]),
        offset_from_anchor_ms=float(used * 1000.0 - ANCHOR_MS[key[0]]),
        offset_agrees_with_anchor=bool(abs(used * 1000.0 - ANCHOR_MS[key[0]])
                                       <= ANCHOR_DISAGREE_MS),
        detections=int(gates['found']), detections_attempted=int(gates['of']),
        face_w=float(gates['face_w']), face_h=float(gates['face_h']),
        face_checks=gates['face_checks'], face_second=int(gates['second_face']),
        face_inside_min=float(gates['face_inside_min']),
        framing_verdict=gates['framing'], app_frac_side_lost=float(gates['app_frac_side_lost']),
        aspect_in_training_range=bool(gates['aspect_in_training_range']),
        motion_verdict=motion, motion_note=motion_note or '',
        motion_crop_px=float((motion_summary or {}).get('amp_median', float('nan'))),
        motion_periodic=float((motion_summary or {}).get('clear_frac', float('nan'))),
        brightness=brightness, frame_ok=bool(FRAME_DARK <= brightness <= FRAME_BRIGHT),
        usable=bool(FRAME_DARK <= brightness <= FRAME_BRIGHT),
        qc_pos_hr=app_read['hr'], qc_pos_status=app_read['status'],
        qc_pos_confidence=app_read['confidence'], qc_pos_kept=app_read['kept'],
        qc_pos_windows=app_read['windows'],
        qc_pos_hr_error=float(app_read['hr'] - ref['metrics']['hr_bpm']),
        qc_pos_bvp_r=pos_r, qc_pos_bvp_lag_s=pos_lag,
        seconds_to_build=round(time.perf_counter() - t_start, 1))
    return arrays, {k: plain(v) for k, v in attrs.items()}


DEMO = ('01', 'motion')              # the shortest recording, so the walkthrough is quick
demo_arrays, demo_attrs = build_record(DEMO)
print('record:', '_'.join(DEMO))
print(json.dumps(demo_attrs, indent=2, default=str))
print({k: (tuple(v.shape), str(v.dtype)) for k, v in demo_arrays.items()})
print(f"NB_P3_35 found this recording's offset at {NB_P3_35_OFFSET_MS[DEMO]:+.1f} ms; this one "
      f"reads {demo_attrs['offset_ms']:+.1f} ms")

record: 01_motion
{
  "schema_version": "p3_v1",
  "dataset": "MR-NIRP-Indoor",
  "role": "heldout",
  "subject_id": "01",
  "subject_uid": "mrnirp_indoor_01",
  "activity": "motion",
  "source_dir": "Subject1_motion_940",
  "fps": 29.95545078389944,
  "fps_nominal": 30.0,
  "fps_median_step": 29.955366503908632,
  "n_frames": 1966,
  "frame_size": 72,
  "source_w": 640,
  "source_h": 640,
  "duration_s": 65.59741044044495,
  "camera_frames_dropped": 0,
  "grid_max_ms": 0.04252239792279511,
  "pc_lag_med_ms": 0.1119375228881836,
  "pc_lag_over_10ms": 0.004069175991861648,
  "crop_rule": "the app box clamped to the frame, from the 8-bit image",
  "box_rx0": 0.018581099808216095,
  "box_ry0": -0.048458438366651535,
  "box_rx1": 1.2475005462765694,
  "box_ry1": 1.1804610081017017,
  "box_side_lost": 0.2013968832439197,
  "bayer_pattern": "RGGB",
  "demosaic": "cv2.COLOR_BayerBG2RGB",
  "sensor_bits": 12,
  "container_bits": 16,
  "byte_order": "big-endian in the file, converted to native 

## 3. Write function and batch processing
Each recording becomes a group named `<subject>_<task>` in one consolidated HDF5, the frames in
`(32, 72, 72, 3)` chunks and uncompressed, every other dataset unchunked. Append mode with
skip-existing makes the run resumable, and each group is flushed as soon as it is written. A
recording reads 1.5-5.9 GB of raw frames, so the full batch is about 48 GB of reads and the store
comes to roughly 1 GiB - one crop, where the Rice store kept two. `MAX_RECORDS` limits a trial
run.

The log goes to `Data/processing_log_mrnirp_indoor_phase3.csv`, and carries recording-level
numbers only - no frames and no waveforms.

In [4]:
LOG_COLS = ('subject_id', 'activity', 'n_frames', 'fps', 'source_w', 'usable', 'frame_ok',
            'detections', 'box_side_lost', 'app_frac_side_lost', 'framing_verdict', 'face_checks',
            'motion_verdict', 'motion_crop_px', 'brightness', 'clipped_share', 'has_hrv',
            'ref_hr_bpm', 'ref_sdnn_ms', 'ref_rmssd_ms', 'ref_kept_share', 'ref_short_by_s',
            'ref_covered_frames', 'offset_ms', 'offset_gap_ms', 'offset_matched',
            'offset_margin_ms', 'offset_from_anchor_ms', 'qc_pos_hr', 'qc_pos_status',
            'qc_pos_hr_error', 'qc_pos_bvp_r', 'qc_pos_bvp_lag_s', 'seconds_to_build',
            'status', 'error')


def write_group(store, name, arrays, attrs):
    """Writes one recording as a group: the frames chunked along time, then every attribute."""
    group = store.create_group(name)
    for key, value in arrays.items():
        if key == 'frames':
            # Whole-frame chunks along time let a clip read touch a few chunks and decompress
            # nothing. The length is clamped because h5py rejects chunks longer than the data.
            chunks = (min(FRAMES_PER_CHUNK, len(value)),) + value.shape[1:]
            group.create_dataset(key, data=value, chunks=chunks)
        else:
            group.create_dataset(key, data=value)
    for key, value in attrs.items():
        group.attrs[key] = value


def process_all(keys, out_h5, limit=None, overwrite=False):
    """Writes every recording into one consolidated HDF5, one group each.

    Append mode with skip-existing makes the run resumable, and a flush after each group keeps
    finished groups on disk if a later one fails.
    """
    keys = list(keys)[:limit] if limit else list(keys)
    rows = []
    out_h5.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    with h5py.File(out_h5, 'a') as store:
        for i, key in enumerate(keys, 1):
            name = f'{key[0]}_{key[1]}'
            if name in store and not overwrite:
                attrs = dict(store[name].attrs)
                attrs.update(status='skipped', error='')
            else:
                try:
                    arrays, attrs = build_record(key)
                    if name in store:
                        del store[name]
                    write_group(store, name, arrays, attrs)
                    store.flush()
                    attrs = dict(attrs, status='ok', error='')
                except Exception as exc:
                    attrs = {'subject_id': key[0], 'activity': key[1], 'usable': False,
                             'status': 'error', 'error': str(exc)}
                print(f'   {i}/{len(keys)} {name} -> {attrs["status"]}'
                      f'{" (" + attrs["error"] + ")" if attrs["error"] else ""} | '
                      f'{(time.perf_counter() - t0) / 60:.1f} min', flush=True)
            rows.append({c: attrs.get(c) for c in LOG_COLS})
    return pd.DataFrame(rows)


ORDER = sorted(REF, key=lambda k: (k[0], TASKS.index(k[1])))
log = process_all(ORDER, OUT_H5, limit=MAX_RECORDS, overwrite=OVERWRITE)
log.to_csv(LOG_PATH, index=False)
print('store:', OUT_H5, f'| {OUT_H5.stat().st_size / 2**30:.2f} GiB')
print('ok:', int((log.status == 'ok').sum()), '| skipped:', int((log.status == 'skipped').sum()),
      '| errors:', int((log.status == 'error').sum()))
print('usable:', int(log.usable.fillna(False).astype(bool).sum()), '/', len(log))
print('log ->', LOG_PATH)

store: D:\QualityPhys\phase 3 benchmarks\mrnirp_indoor_phase3_rc_none.h5 | 0.87 GiB
ok: 0 | skipped: 15 | errors: 0
usable: 15 / 15
log -> D:\code\QualityPhys - CRVSE Project\Data\processing_log_mrnirp_indoor_phase3.csv


## 4. Output validation
Re-opens the store and checks every group against the schema, then the things that make it
trustworthy rather than merely complete:

- **The guards hold.** The training loader's own filter is applied to the store; it must admit
  nothing.
- **NB_P3_35 is reproduced.** The face found in all 180 detection samples, the framing gate
  refusing 13 of 15 and warning on 2, the motion gate accepting all 15, the app's POS reading
  within 1.70 bpm of the reference, 7 references carrying HRV, and each recording's offset back
  where section 7 found it. A difference means the code or the input moved, and nothing
  downstream should run until it is explained.
- **The headroom is re-measurable from the store.** The beats are found again from the stored
  8-bit crops and from `trace12_rgb`, aligned again on the stored reference beats, and the
  full-depth trace has to come out better timed in 13 of the 15 recordings, by a median 6.6 ms
  per recording, as NB_P3_35 section 8's own figures give. Section 8's two medians, 41.6 and
  27.5 ms, are printed beside the store's for reference rather than as a test: both are one
  recording's figures, and that recording's alignment ties across several offsets (section 5).
- **The label sits where it should**: POS against `bvp` negative, at a lag near zero now that
  each recording's offset is applied.

These are checks on the stored data, not an evaluation of any method.

In [5]:
EXPECTED_SETS = {'frames', 'bvp', 'bvp_covered', 'region_rgb', 'trace12_rgb', 'frame_t_ms',
                 'ox_ppg', 'ox_t_ms', 'ox_missing', 'ref_beat_ms', 'ref_beat_kept'}
# NB_P3_35's findings, which this store has to reproduce: sections 6, 7 and 8. Section 8's two
# timing medians are printed for reference only: both are one recording, whose alignment ties
# (section 5), so the comparison that counts is the per-recording one its figures give.
NB35_DETECTIONS, NB35_FRAMING = 180, {'REJECT': 13, 'WARN': 2}
NB35_POS_WORST_BPM, NB35_HRV = 1.70, 7
NB35_TIMING_MS, NB35_TIMING_12BIT_MS = 41.6, 27.5
NB35_CLOSER, NB35_GAIN_MS = 13, 6.6
# Offsets sit on the alignment's own grid, read from the function rather than restated.
OFFSET_STEP_MS = inspect.signature(align_offset).parameters['step_s'].default * 1000.0


def training_filter(group):
    """NB_P3_07's own test: a group it would index must have frames, a bvp and an SQI above 0.05."""
    attrs = group.attrs
    return ('frames' in group and 'bvp' in group
            and float(attrs.get('cardiac_sqi', 0.0)) >= 0.05
            and bool(attrs.get('usable', False))
            and group['frames'].shape[0] >= CLIP_LEN)


def stored_timing_ms(group, trace):
    """The timing error of one trace's beats against the stored reference beats, as NB_P3_35 reads it.

    Read back from the store rather than remembered: the beats are found again from the stored
    trace, aligned again on the stored reference beats, and timed over the beats the alignment
    matched - the median distance among pairs within MATCH_TOL_S, which is section 7's
    `timing_ms` and not the `gap_ms` the alignment itself minimises over every pair.
    """
    fps = float(group.attrs['fps'])
    frame_t = group['frame_t_ms'][:] / 1000.0
    _, beats = camera_beats_on_clock(trace, fps, frame_t)
    ref = group['ref_beat_ms'][:] / 1000.0
    if beats.size < 3:
        return float('nan')
    offset, _, _, _ = align_offset(beats, ref)
    gaps = nearest_gaps(beats + offset, ref)
    close = gaps[gaps <= MATCH_TOL_S]
    return float(np.median(close) * 1000.0) if close.size else float('nan')


with h5py.File(OUT_H5, 'r') as store:
    groups = sorted(store.keys())
    problems = []
    for name in groups:
        group = store[name]
        attrs = group.attrs
        if set(group.keys()) != EXPECTED_SETS:
            problems.append((name, 'datasets', sorted(set(group.keys()) ^ EXPECTED_SETS)))
        n = int(attrs['n_frames'])
        shapes = {k: group[k].shape for k in EXPECTED_SETS}
        if (shapes['frames'] != (n, TARGET_SIZE, TARGET_SIZE, 3)
                or shapes['bvp'] != (n,) or shapes['bvp_covered'] != (n,)
                or shapes['region_rgb'] != (n, len(REGIONS), 3)
                or shapes['trace12_rgb'] != (n, 3) or shapes['frame_t_ms'] != (n,)):
            problems.append((name, 'shapes', shapes))
        if group['frames'].chunks != (FRAMES_PER_CHUNK, TARGET_SIZE, TARGET_SIZE, 3):
            problems.append((name, 'chunks', group['frames'].chunks))
        if 'cardiac_sqi' in attrs or attrs['role'] != 'heldout':
            problems.append((name, 'guard', 'cardiac_sqi present or role not heldout'))
        if group['ref_beat_kept'].shape[0] != group['ref_beat_ms'].shape[0] - 1:
            problems.append((name, 'beats', group['ref_beat_ms'].shape))
        bvp, covered = group['bvp'][:], group['bvp_covered'][:]
        if not np.all(np.isfinite(bvp)) or covered.sum() == 0:
            problems.append((name, 'bvp', 'not finite, or no frame covered'))
    admitted = [name for name in groups if training_filter(store[name])]
    qc = pd.DataFrame([{k: store[name].attrs[k] for k in
                        ('subject_id', 'activity', 'fps', 'source_w', 'usable', 'has_hrv',
                         'detections', 'framing_verdict', 'face_checks', 'motion_verdict',
                         'motion_crop_px', 'box_side_lost', 'brightness', 'clipped_share',
                         'ref_hr_bpm', 'ref_sdnn_ms', 'ref_rmssd_ms', 'ref_kept_share',
                         'ref_short_by_s', 'ref_covered_frames', 'n_frames', 'offset_ms',
                         'offset_gap_ms', 'offset_matched', 'offset_margin_ms',
                         'offset_from_anchor_ms', 'offset_agrees_with_anchor', 'qc_pos_hr',
                         'qc_pos_status', 'qc_pos_hr_error', 'qc_pos_confidence',
                         'qc_pos_bvp_r', 'qc_pos_bvp_lag_s', 'skin_r_g', 'skin_b_g')}
                       for name in groups])
    qc['coverage'] = qc.ref_covered_frames / qc.n_frames
    qc['carried_offset_ms'] = [NB_P3_35_OFFSET_MS[(r.subject_id, r.activity)]
                               for r in qc.itertuples()]
    timing = pd.DataFrame([{'subject_id': store[name].attrs['subject_id'],
                            'activity': store[name].attrs['activity'],
                            'timing_8bit_ms': stored_timing_ms(store[name],
                                                               rgb_trace(store[name]['frames'][:])),
                            'timing_12bit_ms': stored_timing_ms(store[name],
                                                                store[name]['trace12_rgb'][:])}
                          for name in groups])

print(f'groups: {len(groups)} | schema problems: {problems or "none"}')
print(f'the training loader admits {len(admitted)} of {len(groups)} groups '
      f'(it must admit none): {admitted or "none"}')
print(f'usable: {int(qc.usable.sum())}/{len(qc)} | carry HRV: {int(qc.has_hrv.sum())} '
      f'(NB_P3_35: {NB35_HRV}) | face found in all {N_DETECT_FRAMES} samples: '
      f'{int((qc.detections == N_DETECT_FRAMES).sum())}/{len(qc)} '
      f'({int(qc.detections.sum())} of {N_DETECT_FRAMES * len(qc)} samples, NB_P3_35: '
      f'{NB35_DETECTIONS})')
print(f"the app's gates: framing {qc.framing_verdict.value_counts().to_dict()} (NB_P3_35: "
      f'{NB35_FRAMING}), face checks {qc.face_checks.value_counts().to_dict()}, motion '
      f'{qc.motion_verdict.value_counts().to_dict()} ({qc.motion_crop_px.min():.2f}-'
      f'{qc.motion_crop_px.max():.2f} px)')
print(f'the box loses {qc.box_side_lost.min():.0%}-{qc.box_side_lost.max():.0%} of its side '
      f'outside the frame | brightness {qc.brightness.min():.0f}-{qc.brightness.max():.0f} | at '
      f"the sensor's ceiling {qc.clipped_share.min():.3%}-{qc.clipped_share.max():.3%}")
print(f"\nthe app's POS reading against the reference: worst error "
      f'{qc.qc_pos_hr_error.abs().max():.2f} bpm (NB_P3_35: {NB35_POS_WORST_BPM}) | status '
      f'{qc.qc_pos_status.value_counts().to_dict()} | confidence '
      f'{qc.qc_pos_confidence.median():.3f} in the median recording')
print(f'POS against the stored label: r {qc.qc_pos_bvp_r.min():+.2f} to '
      f'{qc.qc_pos_bvp_r.max():+.2f} (negative is the Phase-3 convention), at a lag of '
      f'{qc.qc_pos_bvp_lag_s.min():+.3f} to {qc.qc_pos_bvp_lag_s.max():+.3f} s')
print(f'coverage {qc.coverage.min():.1%}-{qc.coverage.max():.1%} of frames | the reference stops '
      f'{qc.ref_short_by_s.max():.1f} s early at worst')
# Compared in whole grid steps: a 2.5 ms difference computed in floating point lands either side
# of 2.5, so a test on the raw difference counts it one way or the other at random.
steps = ((qc.offset_ms - qc.carried_offset_ms).abs() / OFFSET_STEP_MS).round().astype(int)
print(f'\nalignment: offsets {qc.offset_ms.min():+.0f} to {qc.offset_ms.max():+.0f} ms, '
      f'reproducing NB_P3_35 exactly in {int((steps == 0).sum())}/{len(qc)} recordings and '
      f'within one {OFFSET_STEP_MS:g} ms step in {int((steps <= 1).sum())}/{len(qc)} | '
      f'median distance to a camera beat {qc.offset_gap_ms.median():.1f} ms | the best rival '
      f'alignment is {qc.offset_margin_ms.min():.0f}-{qc.offset_margin_ms.max():.0f} ms worse | '
      f'within {ANCHOR_DISAGREE_MS:.0f} ms of the subject anchor in '
      f'{int(qc.offset_agrees_with_anchor.sum())}/{len(qc)}')
gain = timing.timing_8bit_ms - timing.timing_12bit_ms
print(f'beat timing read back from the store: the full-depth trace closer in '
      f'{int((gain > 0).sum())}/{len(timing)} recordings (NB_P3_35: {NB35_CLOSER}), by a median '
      f'{gain.median():.1f} ms per recording (NB_P3_35: {NB35_GAIN_MS})')
print(f'   medians {timing.timing_8bit_ms.median():.1f} ms from the 8-bit crop and '
      f'{timing.timing_12bit_ms.median():.1f} ms at full depth (section 8: {NB35_TIMING_MS} and '
      f"{NB35_TIMING_12BIT_MS}), for reference only: both are one recording's figures, and its "
      'alignment ties across several grid steps')
print()
print(qc[['subject_id', 'activity', 'fps', 'has_hrv', 'framing_verdict', 'motion_crop_px',
          'brightness', 'ref_hr_bpm', 'qc_pos_hr', 'qc_pos_hr_error', 'qc_pos_bvp_r',
          'offset_ms', 'carried_offset_ms', 'offset_gap_ms', 'coverage']]
      .round(3).to_string(index=False))
print()
print(qc.groupby('activity')
      .agg(recordings=('subject_id', 'size'), hrv=('has_hrv', 'sum'),
           pos_error=('qc_pos_hr_error', lambda v: v.abs().median()),
           confidence=('qc_pos_confidence', 'median'),
           offset_ms=('offset_ms', 'median'), gap_ms=('offset_gap_ms', 'median'),
           motion_px=('motion_crop_px', 'median'))
      .reindex([t for t in TASKS if t in set(qc.activity)]).round(2).to_string())

groups: 15 | schema problems: none
the training loader admits 0 of 15 groups (it must admit none): none
usable: 15/15 | carry HRV: 7 (NB_P3_35: 7) | face found in all 12 samples: 15/15 (180 of 180 samples, NB_P3_35: 180)
the app's gates: framing {'REJECT': 13, 'WARN': 2} (NB_P3_35: {'REJECT': 13, 'WARN': 2}), face checks {'ACCEPT': 15}, motion {'ACCEPT': 15} (0.03-0.12 px)
the box loses 0%-21% of its side outside the frame | brightness 46-120 | at the sensor's ceiling 0.000%-0.556%

the app's POS reading against the reference: worst error 1.72 bpm (NB_P3_35: 1.7) | status {'ok': 13, 'degraded_capture': 2} | confidence 0.807 in the median recording
POS against the stored label: r -0.83 to -0.19 (negative is the Phase-3 convention), at a lag of -0.234 to +0.067 s
coverage 92.5%-100.0% of frames | the reference stops 14.3 s early at worst

alignment: offsets -75 to +233 ms, reproducing NB_P3_35 exactly in 12/15 recordings and within one 2.5 ms step in 14/15 | median distance to a camera b

## 5. Findings (2026-10-28)

**The store.** 15 of 15 recordings written with no errors:
`phase 3 benchmarks\mrnirp_indoor_phase3_rc_none.h5`, 0.87 GiB, 2.4 minutes of build time,
59,269 frames over 8 subjects, every recording usable. Every group passes the schema check:
`frames` at 1,815-7,224 crops of 72x72 in (32, 72, 72, 3) uncompressed chunks, the
frame-aligned arrays at the same length, the oximeter's own record beside them, and the
reference beats with a kept flag on every interval between them. One crop, where the Rice store
kept two, with the full-depth colour trace in place of the second.

**The guards hold.** The training loader's own filter admits 0 of 15: no group carries
`cardiac_sqi`, every group carries `role='heldout'` and `schema_version='p3_v1'`, and the store
sits in `phase 3 benchmarks\`, outside every training glob. The frame clock is strictly
increasing in all 15, every dataset's length agrees with its group's frame count, and every
reference beat falls inside the span of the frames it is dated against.

**The frames are the app's, byte for byte.** One recording was rebuilt from the raw files with
the box the store records, and all 1,966 crops match the stored ones exactly; the held box
itself comes back from a fresh detection pass to the last digit. What the store holds is
therefore what the app's own code produces from this material, not a near copy of it.

**Reproduction.** The face was found in all 180 detection samples, the framing gate refuses 13
and warns on 2, the face checks and the motion gate accept all 15 at 0.03-0.12 px of crop
displacement, and 7 references carry HRV with no disagreement against the set NB_P3_35 named.
The reference itself matches NB_P3_35 section 7 to the three decimals that section printed, in
all 15: heart rate 53.5-81.3 bpm, SDNN 38.8-81.9 ms, RMSSD 25.7-95.0 ms, intervals kept
87.8-100% with a median of 96.5%. The app's POS reading is within 1.72 bpm of the reference in
all 15, against section 6's 1.70; every reading is section 6's scaled by the recording's
measured rate over the nominal 30 fps, to within 0.031 bpm, because the reading now uses the
measured rate.

**The label sits where it should.** POS against the stored `bvp` is negative in all 15 (r -0.83
to -0.19, the Phase-3 convention) at a lag of -0.234 to +0.067 s, with each recording's own
offset already applied.

**Alignment.** Twelve of the fifteen stored offsets are exactly where NB_P3_35 section 7 put
them, two more one 2.5 ms grid step away and 06/still 10 ms away. The three that moved did so
with the rate: the store finds each offset from beats at the recording's measured rate, where
section 7 used the nominal 30 fps, and re-found from the stored crops at the nominal rate all
three land on section 7's offset exactly. Fourteen of fifteen sit within 100 ms of their
subject's anchor; the exception, 07/motion at 320 ms, is the recording section 2 found
anchoring made worse.

**Beat timing, read back from the store.** The full-depth trace is better timed than the 8-bit
crop in 13 of 15 recordings, by a median 6.6 ms per recording - the same 13, and the same 6.6
ms, that NB_P3_35 section 8's figures give. At the nominal rate that section used, the store
reproduces 29 of the 30 entries in its table to the decimal it printed. The one that differs,
01/motion's 8-bit figure, belongs to the median recording at both depths, so both of section
8's medians are its own: 41.6 and 27.5 ms there, 39.1 and 25.1 ms here. Its alignment ties. The
median distance the search minimises is flat to about 2e-13 ms across seven grid points for the
8-bit trace (-25.0 to -10.0 ms), over which the timing statistic reads 28.3 to 41.6 ms, and
exactly flat across four for the full-depth trace (-10.0 to -2.5 ms), over which it reads 21.4
to 27.6 ms. Which point a search returns is decided by rounding for the 8-bit trace (section 8
got -25.0 ms, this readback -22.5) and, for the full-depth one, by the slight change in beats
the measured rate makes (-10.0 at the nominal rate, -7.5 at the measured one). Every one of
those points is optimal under the objective.

**What the full depth buys, restated.** Section 8's 41.6 against 27.5 ms - the "14 ms" and "a
third" carried from it - is a difference of two medians that are both 01/motion, and over that
recording's ties the same difference ranges from about 2 to 20 ms. The robust statements are
per recording, at the nominal rate section 8 used: the full-depth trace is better timed in 13
of 15, still 13 of 15 when each depth is given the worst end of its own tie against the best
end of the other's, by a median 6.6 ms - 4.1-6.6 ms however every tie is broken, a median 24%
of each recording's 8-bit error. The HRV comparison never depended on the alignment, since
intervals are differences of beat times and the offset cancels: from the store at the same rate
the SDNN excess falls from +20.5% to +9.6% and RMSSD's from +66.6% to +45.2%, that section's
figures exactly. Ties appear in 2 of 15 recordings for the 8-bit trace and 3 of 15 for the
full-depth one, and apart from 01/motion they span a single extra grid point and 2.5 ms or
less; they arise where the matched share is low, so a timing figure from such a recording is
not read closer than its tie.

**Nothing is trimmed.** 06/still's reference stops 14.3 s before its last frame, which leaves
one contiguous run of 435 frames at the tail where `bvp_covered` is false and the waveform
holds its last value; coverage is 92.5% there and 99.5-100% in the other fourteen. Marking
rather than trimming keeps the frame count in agreement with the camera logs and matches every
other Phase-3 store, and anything reading `bvp` has to honour `bvp_covered` to use it.

**What the store is for.** The seven still recordings whose references carry HRV keep 165-230 s
of intervals each, about 22 minutes in all - a second read for the SDNN-at-rest result, and the
reason this set was adopted at all. It is not a second camera: the colour camera's setup log
names the Blackfly unit that recorded the Rice set (serial 17473548), through the same
recorder, so the read is independent of Rice in its subjects as far as can be told, its
session, a far darker exposure and 12-bit frames, but not in its camera. SDNN and RMSSD are
offset-free, so the alignment tie cannot reach that evaluation. Two limits travel with it. The
framing gate refuses 13 of the 15 recordings, so every reading from this store is one the app
would not have reported; and the motion recordings move the crop 0.03-0.12 px against a gate
that trips at 1.7 px, so they are not a motion benchmark.